In [5]:
import numpy as np
import time

def m_step_original(gamma, diff, n_k_reciprocal):
    k = diff.shape[2]
    new_covs = np.zeros((k, 2, 2))
    for i in range(k):
        for y in range(diff.shape[0]):
            for x in range(diff.shape[1]):
                new_covs[i] += gamma[y, x, i] * np.outer(diff[y, x, i], diff[y, x, i])
    new_covs *= n_k_reciprocal[:, None, None]
    return new_covs

def m_step_einsum(gamma, diff, n_k_reciprocal):
    new_covs = np.einsum('ijkl,ijkm,ijk->klm', diff, diff, gamma)
    new_covs *= n_k_reciprocal[:, None, None]
    return new_covs

# test data
height, width = 100, 100
k = 5
gamma = np.random.rand(height, width, k)
gamma /= np.sum(gamma, axis=2, keepdims=True)  # 正規化
diff = np.random.randn(height, width, k, 2)
n_k = np.sum(gamma, axis=(0, 1))
n_k_reciprocal = np.reciprocal(n_k)

n_runs = 100  

# original
start_time = time.time()
for _ in range(n_runs):
    result_original = m_step_original(gamma, diff, n_k_reciprocal)
end_time = time.time()
time_original = end_time - start_time

# einsum
start_time = time.time()
for _ in range(n_runs):
    result_einsum = m_step_einsum(gamma, diff, n_k_reciprocal)
end_time = time.time()
time_einsum = end_time - start_time

print(f"originalバージョンの実行時間: {time_original:.6f} 秒")
print(f"einsumバージョンの実行時間: {time_einsum:.6f} 秒")
print(f"速度向上率: {time_original / time_einsum:.2f}倍")

# print(f'result_original:{result_original}')
# print(f'result_einsum:{result_einsum}')

np.testing.assert_allclose(result_original, result_einsum, rtol=1e-5)
print("both res are nearly equal")

originalバージョンの実行時間: 16.558264 秒
einsumバージョンの実行時間: 0.173428 秒
速度向上率: 95.48倍
both res are nearly equal
